# Stage 3: Exploratory Data Analysis (EDA)
## Behavioral Feature Discovery for Human–AI Chess Move Detection

**Project**: EDA-Driven Behavioral Feature Engineering for Human–AI Chess Move Detection  
**Primary Dataset**: `data/processed/clean_chess_moves.csv`  
**Status**: Completed EDA Pipeline  

---

### Objective
Discover repeatable, statistically verified behavioral differences between **HUMAN** and **AI/BOT** players in chess move execution, focusing on thinking time, volatility, game phase dynamics, time pressure reactions, and time-control normalization.


## 1. Setup & Environment Initialization

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import Image, display

DATA_PATH = os.path.join("..", "data", "processed", "clean_chess_moves.csv")
PLOTS_DIR = os.path.join("..", "results", "eda", "plots")
TABLES_DIR = os.path.join("..", "results", "eda", "tables")
STATS_DIR = os.path.join("..", "results", "eda", "statistics")

print("Data path:", os.path.abspath(DATA_PATH))


## 2. First — Data Verification
Load the clean processed dataset and verify integrity, dimensions, missingness, and label breakdown.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape[0]:,d} rows, {df.shape[1]} columns\n")

print("=== Missing Values Analysis ===")
missing = df.isnull().sum()
for col, cnt in missing.items():
    pct = (cnt / len(df)) * 100
    print(f"  {col:20s}: {cnt:5d} null ({pct:.3f}%)")

print("\n=== Class Label Distribution ===")
for lbl, cnt in df["label"].value_counts().items():
    print(f"  {lbl:10s}: {cnt:7,d} ({cnt/len(df)*100:.2f}%)")

print(f"\nUnique Games: {df['game_id'].nunique():,d}")
print(f"Unique Players: {df['player_id'].nunique():,d}")


## 3. Potential Leakage Variables Audit
### Variables NOT Suitable as Predictive Behavioral Features

1. `game_id`: Random 8-character platform session hash. Inclusion causes session memorization and cross-validation leakage.
2. `player_id`: Player username. A model trained on username memorizes specific bots (e.g. `maia1`) rather than learning behavioral physics.
3. `opponent_id`: Opponent username. Memorizes matchmaking pairings.
4. `label`: Target ground-truth classification variable.
5. `white_title` / `black_title`: Platform title badge (`BOT`) used to derive the target label (structural leakage).
6. `game_result`: Final game outcome (`1-0`, `0-1`, `1/2-1/2`), known only post-match.
7. `event`: Tournament and matchmaking header string.


## 4. Univariate Distributions
Inspect the distributions of numerical and categorical variables.

In [ ]:
df_num = pd.read_csv(os.path.join(STATS_DIR, "univariate_numerical.csv"))
display(df_num)

display(Image(filename=os.path.join(PLOTS_DIR, "01_univariate_distributions.png")))


## 5. Human vs. AI Move-Time Comparison
Compare thinking times, variance, and pre-move rates across the two classes.

In [ ]:
df_comp = pd.read_csv(os.path.join(TABLES_DIR, "human_vs_ai_comparison.csv"))
display(df_comp)

display(Image(filename=os.path.join(PLOTS_DIR, "02_human_vs_ai_move_time_dist.png")))
display(Image(filename=os.path.join(PLOTS_DIR, "03_human_vs_ai_zero_moves.png")))


## 6. Move-Time Progression Across the Game
Analyze how human and AI thinking times evolve from move 1 through move 50.

In [ ]:
display(Image(filename=os.path.join(PLOTS_DIR, "04_move_time_progression.png")))


## 7. Player-Level Aggregation & Cognitive Volatility
Examine player-level coefficient of variation (CV) and pre-move percentages.

In [ ]:
df_player = pd.read_csv(os.path.join(TABLES_DIR, "player_level_behavior.csv"))
print("Player-Level Metrics Summary by Class:")
display(df_player.groupby("label")[["mean_move_time", "std_move_time", "cv", "zero_move_rate", "avg_rating"]].describe().T)

display(Image(filename=os.path.join(PLOTS_DIR, "05_player_variability_cv.png")))


## 8. Game Phase Behavior (Opening vs. Middlegame vs. Endgame)
Investigate how tactical complexity and phase transitions alter thinking pacing.

In [ ]:
df_phase = pd.read_csv(os.path.join(TABLES_DIR, "game_phase_behavior.csv"))
display(df_phase)

display(Image(filename=os.path.join(PLOTS_DIR, "06_game_phase_behavior.png")))


## 9. Speed Category & Time-Control Normalization
Compare pacing across Bullet, Blitz, Rapid, and Classical pools, and evaluate normalized budget metrics.

In [ ]:
df_speed = pd.read_csv(os.path.join(TABLES_DIR, "speed_category_behavior.csv"))
display(df_speed)

display(Image(filename=os.path.join(PLOTS_DIR, "07_speed_category_comparison.png")))
display(Image(filename=os.path.join(PLOTS_DIR, "08_time_control_normalization.png")))


## 10. Paired Human-vs-Bot Match Analysis (68 Games)
Evaluate move-level dynamics when Human and Bot play against each other in the exact same game under identical clock conditions.

In [ ]:
df_paired = pd.read_csv(os.path.join(TABLES_DIR, "paired_human_bot_matches.csv"))
display(df_paired)

display(Image(filename=os.path.join(PLOTS_DIR, "09_paired_human_vs_bot_matches.png")))


## 11. Time Pressure Behavioral Divergence (<10s Clock Emergency)
Analyze player reactions as clock reserves deplete into severe time scrambles.

In [ ]:
df_tp = pd.read_csv(os.path.join(TABLES_DIR, "time_pressure_analysis.csv"))
display(df_tp)

display(Image(filename=os.path.join(PLOTS_DIR, "11_time_pressure_behavior.png")))


## 12. Correlation & Outlier Structure
Examine linear and non-linear relationships and extreme thinking pause outliers (>60s).

In [ ]:
display(Image(filename=os.path.join(PLOTS_DIR, "10_correlation_matrix.png")))

with open(os.path.join(STATS_DIR, "outliers_summary.json"), "r") as f:
    outliers = json.load(f)
print("Moves > 60 seconds Summary:")
print(json.dumps(outliers["move_time_gt_60s"], indent=2))


## 13. Statistical Hypothesis Tests Summary

In [ ]:
df_stat = pd.read_csv(os.path.join(TABLES_DIR, "statistical_tests_summary.csv"))
display(df_stat)


## 14. Final EDA Summary

In [ ]:
summary_text = """
==================================================
FINAL EDA SUMMARY
==================================================

Dataset size:
416,978 moves | 6,154 games | 9,812 unique players

Human moves:
402,085 (96.43%)

AI moves:
14,893 (3.57%)

Top behavioral differences:
1. Pre-Move Rate Disparity: AI plays 25.52% of moves at 0.0s (54.70% in Opening) vs. Human 14.78% (26.28% in Opening).
2. Cognitive Volatility vs. Bounded Allocations: Human move-time variance is 2.62x higher in paired matches (std 7.15s vs. 4.42s).
3. Time Pressure Divergence: Under <10s clock scramble, Humans panic-premove at 34.68% with erratic latency jitter (std 2.27s), whereas AI remains composed at 1.64s mean (std 1.57s).
4. Phase Deliberation Asymmetry: Humans surge to 5.29s mean in the Middlegame and collapse to 2.42s in the Endgame from clock starvation; AI pacing is disciplined across all phases.
5. Calculation Pause Outliers: 96.34% of extreme deliberation pauses (>60s) belong to Humans (Human max: 658s vs. AI max: 158s).

Most promising feature candidates:
1. move_time_cv (Rolling move-time Coefficient of Variation)
2. premove_rate (Rolling frequency of 0.0s moves)
3. phase_deliberation_ratio (Middlegame vs. Opening thinking ratio)
4. time_pressure_jitter (Standard deviation of move latency when clock < 15s)
5. relative_move_time (Move time normalized to base time budget T_base / 40)

Important limitations:
1. Class Imbalance: AI moves constitute 3.57% of data; downstream training requires stratification or human downsampling.
2. Speed Pool Heterogeneity: Fast pools (Blitz/Bullet) dominate; raw move times must be normalized to prevent speed confounding.
3. BOT Engine Heterogeneity: Bot accounts range from neural models (Maia) to classical engines (Stockfish), creating multi-modal engine pacing.
==================================================
"""
print(summary_text.strip())
